In [ ]:
# ==========================================
# NLGCL Kaggle 运行指南 (All-in-One: Resume & Upload)
# ==========================================
# 本 Notebook 用于在 Kaggle 环境中配置并运行 NLGCL 模型。
# 
# ## 前置条件
# 1. **GPU 环境**: 请确保 Notebook 设置中 Accelerator 选择 GPU (如 GPU P100 或 T4)。
# 2. **数据集**: 请上传本地 `dataset` 文件夹到 Kaggle Datasets (命名推荐 `nlgcl-dataset`)，并挂载到 Notebook 中。
# 3. **(可选) 恢复训练**: 如需继续训练，请将本地 `saved` 文件夹打包上传为 Dataset，挂载到 Notebook 中。
# 4. **(可选) GitHub Token**: 为了自动上传模型，请在 Kaggle Secrets 中添加名为 `GITHUB_TOKEN` 的密钥。
#
# ------------------------------------------
# STEP 1: 环境检测与代码准备
# ------------------------------------------
import os
import shutil
import glob
import time

# [关键修正] 使用你的个人仓库地址 (确保代码修改能生效)
GITHUB_REPO_URL = "https://github.com/yangzeha/NLGCL.git" 
REPO_NAME = "NLGCL" # 仓库名

# 消除 git 加载日志的辅助函数
def run_silent(cmd):
    os.system(f"{cmd} > /dev/null 2>&1")

# Define IS_KAGGLE global variable
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_KAGGLE:
    # 在 Kaggle 环境下
    # print("Detected Kaggle environment.") # 禁用非必要输出
    
    # 始终回到工作根目录，防止路径混乱
    os.chdir('/kaggle/working')
    
    # [强制清理] 如果目录已存在，先删除，确保 clone 的是最新的正确仓库
    # (之前的运行可能 clone 了别人的旧仓库，导致本地修改无效)
    if os.path.exists(REPO_NAME):
        # print(f"Removing existing {REPO_NAME} directory to force fresh clone...")
        shutil.rmtree(REPO_NAME)
    
    # 克隆代码
    cmd = f"git clone {GITHUB_REPO_URL} {REPO_NAME}"
    # print(f"Executing: {cmd}")
    run_silent(cmd)
    
    # 进入代码目录
    if os.path.exists(REPO_NAME):
        os.chdir(REPO_NAME)
        # print(f"Changed directory to {os.getcwd()}")
        
        # [Kaggle 环境补丁] 创建缺失的目录以防止 recbole_gnn/utils.py 报错
        # 原因: utils.py 会尝试 import sequential_recommender，如果目录不存在则抛出 ModuleNotFoundError
        for missing_module in ["sequential_recommender", "social_recommender"]:
            target_dir = os.path.join("recbole_gnn", "model", missing_module)
            os.makedirs(target_dir, exist_ok=True)
            with open(os.path.join(target_dir, "__init__.py"), "w") as f:
                f.write(f"# {missing_module} module patch for Kaggle\n")
        # print("Applied patches for missing model directories.")

        # [Bug验证修复] 修复NLGCL模型中对比损失未累积的问题
        # 原代码在循环中使用了 = 赋值，导致只保留了最后一层的损失
        # 我们需要将其修改为 += 累加
        nlgcl_model_path = os.path.join("recbole_gnn", "model", "general_recommender", "nlgcl.py")
        if os.path.exists(nlgcl_model_path):
            with open(nlgcl_model_path, 'r', encoding='utf-8') as f:
                content = f.read()
            
            # 定义需要替换的目标字符串 (基于 nlgcl.py 源码)
            # 原始 user loss 赋值
            old_user_loss = "cl_user_loss = self.InfoNCE("
            # 修正 user loss 累加
            new_user_loss = "cl_user_loss += self.InfoNCE("
            
            # 原始 item loss 赋值
            old_item_loss = "cl_item_loss = self.InfoNCE("
             # 修正 item loss 累加
            new_item_loss = "cl_item_loss += self.InfoNCE("
            
            if old_user_loss in content and old_item_loss in content:
                content = content.replace(old_user_loss, new_user_loss)
                content = content.replace(old_item_loss, new_item_loss)
                
                with open(nlgcl_model_path, 'w', encoding='utf-8') as f:
                    f.write(content)
                print(f"Successfully patched {nlgcl_model_path}: Fixed loss accumulation bug.")
            else:
                 # 防止重复patch导致代码错乱，或者版本不对
                if "cl_user_loss +=" in content:
                     print(f"Model {nlgcl_model_path} already patched.")
                else:
                     print(f"Warning: Could not find exact loss assignment pattern in {nlgcl_model_path}. Patch skipped.")
        else:
            print(f"Warning: Model file not found at {nlgcl_model_path}")
            
        # [配置硬修正] 覆写默认模型配置，确保参数生效
        # 为了防止 model/NLGCL.yaml 中的高 cl_reg (1e-5) 覆盖我们的设置，这里直接修改源文件
        nlgcl_config_path = os.path.join("recbole_gnn", "properties", "model", "NLGCL.yaml")
        if os.path.exists(nlgcl_config_path):
             with open(nlgcl_config_path, 'w', encoding='utf-8') as f:
                 # 覆写为 1e-6，平衡 Recall 提升与过拟合风险
                 # 同时将 warm_up_step 设置为 20，让模型尽早（epoch 20）开始利用对比损失
                 # 之前的 yaml 是 40 或 50，可能导致前几十轮根本没有 train_loss3 的梯度（只有计算值）
                 f.write("""
embedding_size: 64
n_layers: 2
reg_weight: 1e-4
cl_temp: 0.2
alpha: 0.4
cl_reg: 1e-6
warm_up_step: 20
""")
             print(f"Successfully updated {nlgcl_config_path} with optimized cl_reg=1e-6 and warm_up_step=20")

else:
    print("Not in Kaggle environment, assuming local run.")
    # 如果在本地，不需要 clone，直接运行即可

# ------------------------------------------
# STEP 2: 安装依赖 (极速版 - 优化下载体积)
# ------------------------------------------
# 为了解决 Kaggle 预装 PyTorch 版本过新导致的 PyG 编译缓慢问题，
# 我们降级到 PyTorch 2.5.1 以利用官方二进制包 (Whl)。
# 这个过程需要下载约 800MB 数据，在 Kaggle 网络下通常需要 1-2 分钟。
# 这是目前最快且最稳定的方案。

print("System initializing... (Please wait 1-2 minutes)")

# 使用 %%capture 隐藏冗长的安装日志，只显示关键信息
# 注意：在 Notebook 代码单元格中使用 %%capture 需要放在最开头，
# 这里我们用 Python 上下文管理器来模拟，或者直接忍受日志。
# 为了直观，我们保留日志但精简安装列表。

# 1. [核心修复] 降级 NumPy < 2.0
# RecBole 和旧版代码不兼容 NumPy 2.0 (移除了 np.float_ 等属性)
# print("Downgrading NumPy to < 2.0 for compatibility...")
!pip install -q "numpy<2.0" > /dev/null 2>&1

# 2. 安装 RecBole (会自动处理依赖，忽略 colorama 警告即可)
# 使用 -q (quiet) 减少输出
!pip install -q --upgrade recbole > /dev/null 2>&1

# 3. 这里的策略是：只安装 torch 核心包，不安装 torchvision/torchaudio 以节省时间和流量
#    NLGCL 是图推荐模型，通常不需要视觉和音频库
# print("Switching to stable PyTorch 2.5.1 (Core only) for fast setup...")
!pip install -q torch==2.5.1+cu124 --extra-index-url https://download.pytorch.org/whl/cu124 > /dev/null 2>&1

# 4. 秒装 PyG 二进制包
# print("Installing PyG binaries...")
wheel_url = "https://data.pyg.org/whl/torch-2.5.1+cu124.html"
!pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv -f {wheel_url} > /dev/null 2>&1
!pip install -q torch_geometric > /dev/null 2>&1

# 5. 安装其他依赖
!pip install -q -r requirements.txt > /dev/null 2>&1

import torch
import numpy as np
# print(f"Final PyTorch Version: {torch.__version__}")
# print(f"Final NumPy Version: {np.__version__}")
# print(f"CUDA Available: {torch.cuda.is_available()}")
print("Environment setup complete.")

# ------------------------------------------
# STEP 3: 数据集准备 (自动寻找与链接)
# ------------------------------------------
# 自动寻找 Kaggle Input 中的 dataset 目录
def find_dataset_root(search_path='/kaggle/input'):
    for root, dirs, files in os.walk(search_path):
        if 'yelp.inter' in files:
            if os.path.basename(root) == 'yelp':
                return os.path.dirname(root)
            return os.path.dirname(root)
    return None

if IS_KAGGLE:
    current_dataset_link = 'dataset'
    
    # 1. 寻找挂载的数据集
    found_path = find_dataset_root()
    if not found_path:
        # 尝试默认路径
        potential_paths = [
            "/kaggle/input/nlgcl-dataset",
            "/kaggle/input/nlgcl-dataset/dataset",
            "/kaggle/input/dataset"
        ]
        for p in potential_paths:
            if os.path.exists(os.path.join(p, 'yelp', 'yelp.inter')):
                found_path = p
                break
    
    if found_path:
        # print(f"Found dataset at: {found_path}")
        
        # 2. 清理旧的 dataset 目录/链接
        if os.path.exists(current_dataset_link):
            if os.path.islink(current_dataset_link):
                os.unlink(current_dataset_link)
            else:
                shutil.rmtree(current_dataset_link)
        
        # 3. 复制数据集
        # print("Copying dataset to working directory (required for write access)...")
        shutil.copytree(found_path, current_dataset_link)
        # print("Dataset copied successfully.")
        
    else:
        print("Error: Could not find 'yelp/yelp.inter' in /kaggle/input.")
        !find /kaggle/input -maxdepth 3
else:
    print("Local environment, skipping dataset setup.")

# !ls -l dataset/yelp/

# ------------------------------------------
# STEP 4: 强制写入过滤配置 (覆盖 yelp.yaml)
# ------------------------------------------
# [续训修改] 将 epochs 从 300 提高到 600 以支持继续训练
# 如果是从头开始，这会跑 600 轮；
# 如果是 Resume，这会继续跑直到总轮数达到 600。
yelp_config_content = """
load_col:
  inter: [user_id, item_id, rating]
ITEM_ID_FIELD: item_id
RATING_FIELD: rating

# 核心过滤配置：过滤交互少于 15 次的用户和物品
user_inter_num_interval: "[15,inf)"
item_inter_num_interval: "[15,inf)"

val_interval:
  rating: "[3,inf)"

# ----------------
# 训练与日志配置 (已优化速度)
# ----------------
epochs: 600
train_batch_size: 4096
learner: adam
learning_rate: 0.001

# [速度优化] 将评估间隔改为 5 (从每轮评估变为5轮评估一次)
eval_step: 5
# [速度优化] 增大评估 Batch Size (GPU T4/P100 显存足够)
# Full Ranking 非常耗时，增大此值可显著通过并行度加快评估
eval_batch_size: 40960

stopping_step: 200
# 评估指标
metrics: ["Recall", "NDCG"]
topk: [10, 20, 50]
valid_metric: NDCG@10

# [调参优化]
# 之前的 1e-7 导致 Loss 过小 (~2.0)，效果回升慢
# 之前的 1e-5 (implied) 导致 Loss 过大 (~180)，效果崩塌
# 折中方案：1e-6。预期 Loss ~20
# 注意：主配置文件的 warm_up_step 设置为 20，这里 40 可能会被覆盖或需要保持一致
cl_reg: 1e-6
warm_up_step: 20

# 日志输出控制
verbose: True
state: INFO
show_progress: False  # 禁止显示进度条，只保留日志输出
"""

os.makedirs('properties', exist_ok=True)
with open('properties/yelp.yaml', 'w') as f:
    f.write(yelp_config_content)
# print("Updated properties/yelp.yaml with filtering rules and removed progress bar.")

# ------------------------------------------
# STEP 5: 恢复存档 & 运行训练
# ------------------------------------------
import subprocess
import re
import sys
import torch # Ensure torch is imported for validation

# A. 搜索并恢复 Kaggle Input 中的 Checkpoint
# ------------------------------------------------------------------
# 当用户上传了一个 saved 文件夹（无论是zip解压还是直接文件夹）作为 Dataset 时，
# 它们通常位于 /kaggle/input/xxxx/saved/xxx.pth 或 /kaggle/input/xxxx/xxx.pth
if IS_KAGGLE:
    # 强制创建 saved 文件夹
    os.makedirs("saved", exist_ok=True)
    
    print("[INIT] Searching for uploaded checkpoints in /kaggle/input...")
    
    # 优先搜索用户指定的 'nlgcl-saved' 或包含 'model' 的路径
    candidates = []
    
    # 添加防呆逻辑：如果用户上传了 saved 文件夹，里面可能混杂了 dataset 缓存 (pickle) 和 模型权重 (pth)
    # 我们只复制模型权重，坚决不复制 dataset 缓存，因为 pickle 文件极易跨环境不兼容导致 UnpicklingError
    
    target_dirs = ["/kaggle/input/nlgcl-saved/saved", "/kaggle/input/nlgcl-saved", "/kaggle/input"]
    
    found_ckpt = False
    
    for search_dir in target_dirs:
        if os.path.exists(search_dir):
            for root, dirs, files in os.walk(search_dir):
                for f in files: # Case insensitive check for dataset
                    # [关键逻辑] 
                    # 1. 必须是 .pth 结尾
                    # 2. 排除 'dataset' 关键字 (大小写敏感或不敏感)
                    # 3. 排除 'dataloader' 关键字
                    f_lower = f.lower()
                    if f.endswith(".pth") and "dataloader" not in f_lower and "dataset" not in f_lower:
                         # 只要文件名不含 dataset (比如 yelp-GeneralGraphDataset.pth 就会被排除)
                         # 这样才能确保 resume_file 只是模型权重
                         full_path = os.path.join(root, f)
                         candidates.append(full_path)
        if candidates: 
            break # 找到了就停止大范围搜索
            
    if not candidates:
        # Fallback: 全局搜索
        for root, dirs, files in os.walk("/kaggle/input"):
             for f in files:
                f_lower = f.lower()
                if f.endswith(".pth") and "dataloader" not in f_lower and "dataset" not in f_lower:
                     if "NLGCL" in f: # 只信任带模型名的
                        candidates.append(os.path.join(root, f))

    if candidates:
        # 排序取最新
        candidates.sort(key=lambda x: os.path.getmtime(x) if os.path.exists(x) else x)
        target_ckpt = candidates[-1]
        print(f"[RESTORE] Found checkpoint: {target_ckpt}")
        
        dest_path = os.path.join("saved", os.path.basename(target_ckpt))
        if not os.path.exists(dest_path):
            shutil.copy2(target_ckpt, dest_path)
            print(f"[RESTORE] Copied to workspace: {dest_path}")
        else:
            print(f"[RESTORE] Checkpoint already present in workspace.")
    else:
        print("[INIT] No valid model checkpoint found (ignored dataset caches). Starting fresh training.")

    # [清理] 再次确保 saved 目录下没有 dataset 缓存文件
    # 任何 saved/*dataset*.pth (包含 yelp-GeneralGraphDataset.pth) 都会导致 RecBole 尝试 pickle.load 从而报错
    # 必须强制删除所有疑似 dataset 的文件
    for f in glob.glob("saved/*.pth"):
        if "dataset" in f.lower():
            try:
                os.remove(f)
                print(f"[CLEANUP] Removed potentially corrupted dataset cache: {f}")
            except Exception as e:
                print(f"[CLEANUP] Failed to remove {f}: {e}")

# B. 生成运行命令 (含 resume 参数)
# ------------------------------------------------------------------
train_cmd = ["python", "-u", "main.py", "--dataset=yelp"]

# 再次检查 saved 目录（包含刚刚恢复的）
resume_previous_success = False

if os.path.exists("saved"):
    # 同样只找模型文件
    # 这里的 filter 必须跟上面保持一致
    pth_files = [f for f in glob.glob("saved/*.pth") if "dataloader" not in f.lower() and "dataset" not in f.lower()]
    if pth_files:
        # 找最新的
        candidate_resume_file = max(pth_files, key=os.path.getmtime)
        
        # [新增] 校验文件是否为有效的 PyTorch Checkpoint
        # 很多时候 git clone 下来的大文件是 LFS 指针 (文本文件)，会导致 UnpicklingError: invalid load key, 'v'
        print(f"[RESUME] Validating checkpoint: {candidate_resume_file}...")
        try:
            # 尝试加载头部 (map_location='cpu' 避免占用太多显存)
            # 注意: 这会加载整个文件到内存，如果文件很大可能会稍慢，但比运行报错要好
            dummy = torch.load(candidate_resume_file, map_location='cpu')
            del dummy
            print(f"[RESUME] Checkpoint is valid. Resuming training...")
            train_cmd.extend(["--resume_file", candidate_resume_file])
            resume_previous_success = True
        except Exception as e:
            print(f"[RESUME] CRITICAL ERROR: Checkpoint validation failed!")
            print(f"         File: {candidate_resume_file}")
            # print(f"         Error details: {e}")
            
            # [深度诊断 + 自动修复]
            try:
                file_size_mb = os.path.getsize(candidate_resume_file) / (1024 * 1024)
                print(f"         File size: {file_size_mb:.2f} MB")
                
                with open(candidate_resume_file, 'rb') as f:
                    header_bytes = f.read(64)
                
                header_str = header_bytes.decode('utf-8', errors='ignore')
                
                if "version https://git-lfs" in header_str:
                     print("         [CONCLUSION] This is a Git LFS pointer (Text file).")
                     
                     # --- [AUTO-FIX] 自动从 GitHub 下载原文件 ---
                     # 构造 Raw URL: https://github.com/yangzeha/NLGCL/raw/main/saved/NLGCL-Mar-16-2026_15-49-01.pth
                     filename = os.path.basename(candidate_resume_file)
                     # 注意：这里假设 branch 是 main
                     raw_url = f"https://github.com/yangzeha/NLGCL/raw/main/saved/{filename}"
                     print(f"         [AUTO-FIX] Attempting to download REAL file from GitHub: {raw_url}")
                     
                     # 使用 wget 下载覆盖
                     # -O 覆盖原文件
                     ret_code = os.system(f"wget -q -O {candidate_resume_file} {raw_url}")
                     
                     if ret_code == 0:
                         new_size = os.path.getsize(candidate_resume_file) / (1024 * 1024)
                         print(f"         [AUTO-FIX] Download successful. New size: {new_size:.2f} MB")
                         
                         if new_size > 1: # 稍微大于1MB就认为是二进制了
                             print(f"         [AUTO-FIX] File looks good! Retrying resume...")
                             train_cmd.extend(["--resume_file", candidate_resume_file])
                             resume_previous_success = True
                         else:
                             print(f"         [AUTO-FIX] Downloaded file is still too small. GitHub raw link might be wrong.")
                     else:
                          print(f"         [AUTO-FIX] Wget failed with code {ret_code}")

                else:
                     print("         [CONCLUSION] File corrupted (Not LFS).")
                     
            except Exception as diag_e:
                print(f"         [Diagnose Error] {diag_e}")
            
            if not resume_previous_success:
                print("         (Starting fresh training due to invalid checkpoint)")

# C. 启动训练进程
# ------------------------------------------------------------------
print(f"Starting command: {' '.join(train_cmd)}")

print_enabled = False
# 正则匹配损失行
loss_pattern = re.compile(r"train_loss1: ([\d\.]+), train_loss2: ([\d\.]+), train_loss3: ([\d\.]+)")

with subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
    for line in p.stdout:
        # 过滤逻辑：
        # 1. 正常训练: 检测 "The number of users"
        # 2. 恢复训练: 可能会跳过数据加载日志，直接显示 "Loading model structure" 或 "batch_size ="
        if any(k in line for k in ["The number of users", "Loading model", "resume_checkpoint", "train_batch_size"]):
            print_enabled = True
        
        # 始终显示的关键信息
        if "Loading model" in line or "resume" in line or "epoch" in line:
             pass # 这些行在下面打印，只要 print_enabled=True

        if print_enabled:
            # 过滤掉杂乱的进度条字符
            if "GPU RAM" in line or "%" in line: continue
                
            # 计算并显示 Total Loss
            match = loss_pattern.search(line)
            if match:
                try:
                    l1, l2, l3_val = match.groups()
                    total = float(l1) + float(l2) + float(l3_val)
                    line = line.strip()
                    if line.endswith(']'):
                        line = line[:-1] + f", total_loss: {total:.4f}]"
                    else:
                        line = line + f" (Total Loss: {total:.4f})"
                    print(line)
                except ValueError:
                    print(line, end='')
            else:
                print(line, end='')

if p.returncode != 0:
    print(f"\nTraining finished with return code {p.returncode}")
else:
    print("\nTraining completed successfully.")